# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Danishh-ux/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Ranking / scoring.**

My question from Week 1 was "which pages should be reviewed first?" — that's a "which ones
first" question, which the framing skill maps directly to **ranking/scoring**, not plain
classification. I'm not just trying to answer yes/no "is this page declining" (that's a
sub-question I can use as an ingredient); I need an ordered queue an editor can work down from
the top. A classifier alone gives me a bucket (declining / not declining) with no way to say
which of the ~16,000 declining pages to look at *first* — scoring gives me that order.

In practice this will still lean on a classification-flavored target underneath (see section
2), but the deliverable a content editor actually uses is a **ranked list**, so I'm framing the
task as ranking/scoring, with a supervised score under the hood.

In [ ]:
# Confirming the reference metric helper exists and behaves as expected on toy data
import sys
sys.path.insert(0, "../../scripts")
from ml_utils import precision_at_k

# toy check: perfect ranking should give precision@k = 1.0 when all top-k are positives
y_true = [1, 1, 1, 0, 0]
scores = [0.9, 0.8, 0.7, 0.2, 0.1]
print("Precision@3 on a perfect ranking:", precision_at_k(y_true, scores, k=3))


Precision@3 on a perfect ranking: 1.0


## 2. Target or proxy

**Target: `is_declining_label`** — 1 when `trend_direction == "down"`, else 0. This is the
same label the reference pipeline (`scripts/01_prepare_features.py`) defines.

**Where it comes from — observed, not hand-defined by me:** `trend_direction` itself is
computed from `trend_pct`, which compares `impressions_last_30d` against `impressions_prev_30d`
(an actual measured swing in search impressions, not a rule I invented). So the label is an
**observed outcome** — a real change in demand that already happened — not a proxy I made up
after the fact.

**The leakage trap I have to respect:** because the label is *derived from* `trend_direction`
(and that in turn from `trend_pct`), neither `trend_direction` nor `trend_pct` can ever be a
model feature — using them would mean the model is just learning to reverse-engineer its own
label. I'll build my feature vector only from signals that exist independently of that trend
window (content properties, activity totals, position, engagement) — this is exactly the
leakage check I'll formalize in Week 3.

For the final **ranking**, my score won't be the raw label alone — it'll combine
`is_declining_label` with a demand signal (e.g. `impressions_90d`), since Week 1's numbers
showed that "declining" alone is too broad (54.2% of pages) to be a useful priority list on
its own.

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# The target: observed, not hand-defined -- derived straight from trend_direction
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())
print(f"\nBase rate (share declining): {df['is_declining_label'].mean():.1%}")

# A few real rows so the label is concrete, not abstract
df[["content_id", "client_id", "trend_direction", "trend_pct", "impressions_90d",
    "is_declining_label"]].sample(5, random_state=42)


is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Base rate (share declining): 54.2%


,content_id,client_id,trend_direction,trend_pct,impressions_90d,is_declining_label
2308,content_9824710082d8,client_3fdba35f04,stable,8.0,283,0
22404,content_3efa3a7c46bb,client_f74efabef1,up,22.9,8878,0
23397,content_575dc8a2ab0f,client_25fc0e7096,new,NaN,3,0
25058,content_0dbd6911ba04,client_d029fa3a95,down,-57.4,124,1
2664,content_bbaf87019afb,client_19581e27de,down,-20.8,4294,1


## 3. Success metric

**Primary metric: Precision@K (specifically Precision@50), matching the reference pipeline's
`precision_at_k` helper in `scripts/ml_utils.py`.**

Why this metric fits the decision from Week 1: an editor works down a ranked list starting at
the top and has limited time — what matters is "of the pages I actually get reviewed (the
top K), how many were worth reviewing?" That is precisely what Precision@K measures. A
global metric like plain accuracy or ROC-AUC would reward the model for being right about the
*bulk* of pages (including the ones near the bottom nobody will ever look at), which isn't
what I care about.

**What "good" means for this task:** the committed reference report (`outputs/model_report.md`)
shows baseline rule Precision@50 ≈ 0.24 vs a trained model reaching ≈ 0.68–0.74 — roughly a 3x
lift. I'll use that same baseline-vs-model comparison as my own bar for "good" once I build my
own baseline (Week 4) and model (Week 5).

In [ ]:
# Sanity-checking Precision@K against the committed reference numbers
# (outputs/model_report.md reports baseline P@50 ~= 0.24, model P@50 ~= 0.68-0.74)
import json

report_path = "../../outputs/model_report.md"
with open(report_path) as f:
    text = f.read()

# just confirm the file exists and mentions Precision@50 -- I'll build my own version
# of this comparison in Weeks 4-5 using my own baseline and model
print("Precision@50" in text, "-- 'Precision@50' appears in the committed reference report")
print("\nThis confirms the metric I'm committing to is the same one the reference pipeline")
print("already reports on, so my Week 4/5 numbers will be directly comparable to it.")


True -- 'Precision@50' appears in the committed reference report

This confirms the metric I'm committing to is the same one the reference pipeline
already reports on, so my Week 4/5 numbers will be directly comparable to it.


## 4. The unit of analysis, as a real dataframe

One row = one content item (page), identified by `content_id`, belonging to one `client_id`.
Loading the lane's slice below and showing it as an actual dataframe, plus a first sketch of
what the target column looks like on real rows.

In [ ]:
# Unit of analysis: one row = one page (content_id), nested under one client (client_id)
print(f"Rows: {len(df):,} | Unique content_id: {df['content_id'].nunique():,} "
      f"| Unique client_id: {df['client_id'].nunique()}")

df[["content_id", "client_id", "content_type", "word_count", "impressions_90d",
    "avg_position", "freshness_tier", "trend_direction", "is_declining_label"]].head(5)


Rows: 30,000 | Unique content_id: 30,000 | Unique client_id: 32


,content_id,client_id,content_type,word_count,impressions_90d,avg_position,freshness_tier,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,3803,10.6,0-30,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,15320,20.3,0-30,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,12581,36.5,0-30,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,11751,6.2,0-30,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,19140,44.0,0-30,down,1


## 5. Why ML beats a fixed rule here

A fixed rule (e.g. "flag every page where `trend_direction == 'down'`") is a real, working
baseline — and I'll actually build and keep that exact baseline in Week 4, not throw it away.
But Week 1's numbers already show its weakness: **54.2%** of all pages are "declining" by that
rule, while only **37.2%** of pages are declining *and* still carry real search demand
(`impressions_90d >= 300`). A single if-statement can't easily balance "how much did it drop"
against "how much traffic is actually at stake" against "how visible was it to begin with"
(`position_tier`) against "is it stale anyway" (`freshness_tier`) — the moment I want to weigh
several signals against each other instead of gating on one, I'm no longer writing an
if-statement, I'm learning weights. That's the exact situation the framing skill describes as
"the pattern is real but too messy to write by hand — many signals, tangled, shifting over
time."

The section below shows this concretely: even restricting to declining pages, demand
(`impressions_90d`) and content freshness don't move together in any simple, hand-codable
way — which is the messiness a learned score is meant to sort through.

In [ ]:
# Showing the "too messy for an if-statement" claim concretely:
# among DECLINING pages only, demand (impressions) and staleness (freshness_tier)
# don't line up in any simple, single-threshold way.
declining = df[df["is_declining_label"] == 1]

messy = declining.groupby("freshness_tier")["impressions_90d"].agg(["count", "median", "mean"])
print(messy)

print("\nIf staleness alone predicted 'worth fixing', median impressions would rise cleanly")
print("with freshness_tier. It doesn't move in a clean single direction across tiers --")
print("which is exactly why a one-column if-statement can't capture this, but a model that")
print("weighs several signals together (freshness + demand + position + content type) can.")


                count  median         mean
freshness_tier                            
0-30            10473   713.0  4238.859926
181+               82    39.0  2435.231707
31-90             103   427.0  2628.339806
91-180           5604  1595.0  6268.803533

If staleness alone predicted 'worth fixing', median impressions would rise cleanly
with freshness_tier. It doesn't move in a clean single direction across tiers --
which is exactly why a one-column if-statement can't capture this, but a model that
weighs several signals together (freshness + demand + position + content type) can.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.